In [1]:
"""
=============================================================================
Finansal Olay Calismasi Analizi — Moduler Veri Boru Hatti

Yazar   : Veri Muhendisi Boru Hatti (Data Engineer Pipeline)
Surum   : 1.0.0
Cikti   : sp500_olay_calismasi_ham_veri.csv  (Uzun Format, Modele Hazir)

Bagimliliklar:
    pip install yfinance pandas numpy

Kullanim:
    python olay_calismasi_boru_hatti.py
"""

# HÜCRE 1: Kütüphaneler ve Ayarlar
from __future__ import annotations

import logging
import sys
import time
from dataclasses import dataclass, field
from datetime import date, timedelta
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger(__name__)

print("Hücre 1 çalıştı: Kütüphaneler yüklendi ve loglama ayarlandı.")

Hücre 1 çalıştı: Kütüphaneler yüklendi ve loglama ayarlandı.


In [2]:
# HÜCRE 2: Veri Sınıfları ve Yapılandırma

@dataclass(frozen=True)
class Olay:
    """Sifir-tarihi (t0) ile tek bir makro olayi temsil eder."""
    isim: str
    t0: date

@dataclass(frozen=True)
class HisseBilgisi:
    """Bir hisse sembolunu analitik grup etiketine esler."""
    sembol: str
    grup: str

@dataclass
class BoruHattiAyarlari:
    """kullanilan merkezi yapilandirma nesnesi."""
    tahmin_gunleri: int = 252
    olay_gunleri: int = 60
    isinma_gunleri: int = 40
    vol_kisa: int = 10
    vol_uzun: int = 30
    cikti_dosyasi: str = "sp500_olay_calismasi_ham_veri.csv"

    olaylar: List[Olay] = field(default_factory=lambda: [
        Olay("ABD_Secimleri_2016",        date(2016, 11,  8)),
        Olay("COVID19_Pandemisi_2020",    date(2020,  3, 11)),
        Olay("Rusya_Ukrayna_Savasi_2022", date(2022,  2, 24)),
        Olay("Israil_Iran_2024",          date(2024,  4, 15)),
        Olay("ABD_Secimleri_2024",        date(2024, 11,  5)),
    ])

    hisseler: List[HisseBilgisi] = field(default_factory=lambda: [
        HisseBilgisi("SPY",  "Gosterge_Endeks"),
        HisseBilgisi("^VIX",   "Gosterge_Endeks"),
        HisseBilgisi("XLF",    "Odak_Sektor"),
        HisseBilgisi("XLK",    "Odak_Sektor"),
        HisseBilgisi("XLE",    "Odak_Sektor"),
        HisseBilgisi("XLV",    "Odak_Sektor"),
        HisseBilgisi("ITA",    "Odak_Sektor"),
        HisseBilgisi("XLP",    "Kontrol_Grubu"),
        HisseBilgisi("XLRE",   "Kontrol_Grubu"),
        HisseBilgisi("XLY",    "Kontrol_Grubu"),
        HisseBilgisi("GLD",    "Emtia_Makro"),
        HisseBilgisi("CL=F",   "Emtia_Makro"),
    ])

print("Hücre 2 çalıştı: Parametreler ve Olay/Hisse tanımlamaları yapıldı.")

Hücre 2 çalıştı: Parametreler ve Olay/Hisse tanımlamaları yapıldı.


In [3]:
# HÜCRE 3: Veri Muhendisi Sınıfının Tanımlanması

class VeriMuhendisi:
    def __init__(self, ayar: Optional[BoruHattiAyarlari] = None) -> None:
        self.ayar = ayar or BoruHattiAyarlari()
        self._ham_onbellek: Dict[str, pd.DataFrame] = {}   
        self._uzun_cerceveler: List[pd.DataFrame] = []

    def calistir(self) -> pd.DataFrame:
        log.info("=" * 70)
        log.info("Olay Calismasi Boru Hatti — BASLADI")
        log.info("=" * 70)

        tum_semboller = [hisse.sembol for hisse in self.ayar.hisseler]
        self._tumunu_onceden_indir(tum_semboller)

        for olay in self.ayar.olaylar:
            log.info("Olay isleniyor: %s  (t0=%s)", olay.isim, olay.t0)
            indirme_baslangici, olay_bitisi = self._pencere_tarihlerini_hesapla(olay.t0)

            for hisse_bilgisi in self.ayar.hisseler:
                cerceve = self._pencere_kes(
                    sembol             = hisse_bilgisi.sembol,
                    indirme_baslangici = indirme_baslangici,
                    olay_bitisi        = olay_bitisi,
                    t0                 = olay.t0,
                    olay_ismi          = olay.isim,
                    grup               = hisse_bilgisi.grup,
                )
                if cerceve is not None and not cerceve.empty:
                    self._uzun_cerceveler.append(cerceve)
                else:
                    log.warning("  %s / %s icin kullanilabilir veri yok", olay.isim, hisse_bilgisi.sembol)

        if not self._uzun_cerceveler:
            raise RuntimeError("Sifir satir uretildi.")

        log.info("Tum dilimler birlestiriliyor ...")
        df = pd.concat(self._uzun_cerceveler, ignore_index=True)
        df = self._eksik_verileri_isle(df)
        df = self._son_hali_ver(df)

        df.to_csv(self.ayar.cikti_dosyasi, index=False)
        log.info("CSV kaydedildi -> %s  (%d satir x %d sutun)", self.ayar.cikti_dosyasi, *df.shape)
        log.info("Boru Hatti — TAMAMLANDI")

        return df

    def _pencere_tarihlerini_hesapla(self, t0: date) -> Tuple[date, date]:
        takvim_orani = 1 / 0.7
        tahmin_takvim = int((self.ayar.tahmin_gunleri + self.ayar.isinma_gunleri) * takvim_orani)
        olay_takvim   = int(self.ayar.olay_gunleri * takvim_orani)
        return t0 - timedelta(days=tahmin_takvim), t0 + timedelta(days=olay_takvim)

    def _tumunu_onceden_indir(self, semboller: List[str]) -> None:
        log.info("%d hisse icin ham veri indiriliyor ...", len(semboller))
        baslangiclar, bitisler = [], []
        for olay in self.ayar.olaylar:
            bas, bit = self._pencere_tarihlerini_hesapla(olay.t0)
            baslangiclar.append(bas)
            bitisler.append(bit)

        genel_baslangic = min(baslangiclar)
        genel_bitis     = max(bitisler)

        for sembol in semboller:
            log.info("  %s indiriliyor ...", sembol)
            try:
                ham_veri = yf.download(
                    tickers    = sembol,
                    start      = genel_baslangic.strftime("%Y-%m-%d"),
                    end        = (genel_bitis + timedelta(days=1)).strftime("%Y-%m-%d"),
                    auto_adjust= True,
                    progress   = False,
                    threads    = False,
                )
                if ham_veri.empty:
                    self._ham_onbellek[sembol] = pd.DataFrame()
                else:
                    if isinstance(ham_veri.columns, pd.MultiIndex):
                        ham_veri.columns = ham_veri.columns.get_level_values(0)
                    ham_veri.index = pd.to_datetime(ham_veri.index).tz_localize(None)
                    self._ham_onbellek[sembol] = ham_veri
            except Exception as hata:
                log.error("  %s indirilirken HATA: %s", sembol, hata)
                self._ham_onbellek[sembol] = pd.DataFrame()
            time.sleep(0.3)

    def _pencere_kes(self, sembol, indirme_baslangici, olay_bitisi, t0, olay_ismi, grup) -> pd.DataFrame:
        ham_veri = self._ham_onbellek.get(sembol)
        if ham_veri is None or ham_veri.empty:
            return None

        maske = (ham_veri.index >= pd.Timestamp(indirme_baslangici)) & (ham_veri.index <= pd.Timestamp(olay_bitisi))
        df = ham_veri.loc[maske, ["Close"]].copy()
        
        if df.empty: return None

        df.rename(columns={"Close": "Duzeltilmis_Kapanis"}, inplace=True)
        
        tam_indeks = pd.date_range(df.index.min(), df.index.max(), freq="B")
        df = df.reindex(tam_indeks).ffill()
        df.index.name = "Tarih"

        df["Log_Getiri"] = np.log(df["Duzeltilmis_Kapanis"] / df["Duzeltilmis_Kapanis"].shift(1))
        df["Volatilite_10g"] = df["Log_Getiri"].rolling(window=self.ayar.vol_kisa, min_periods=self.ayar.vol_kisa).std()
        df["Volatilite_30g"] = df["Log_Getiri"].rolling(window=self.ayar.vol_uzun, min_periods=self.ayar.vol_uzun).std()

        takvim_orani = 1 / 0.7
        kirpma_baslangici = pd.Timestamp(t0) - timedelta(days=int(self.ayar.tahmin_gunleri * takvim_orani))
        df = df.loc[(df.index >= kirpma_baslangici) & (df.index <= pd.Timestamp(olay_bitisi))]

        if df.empty: return None

        df = df.reset_index()
        df["Olay_Ismi"] = olay_ismi
        df["Hisse"]     = sembol
        df["Grup"]      = grup
        df["T0_Goreceli_Gun"] = (df["Tarih"] - pd.Timestamp(t0)).dt.days

        return df

    def _eksik_verileri_isle(self, df: pd.DataFrame) -> pd.DataFrame:
        sayisal_sutunlar = ["Duzeltilmis_Kapanis", "Log_Getiri", "Volatilite_10g", "Volatilite_30g"]
        df = (
            df.sort_values(["Olay_Ismi", "Hisse", "Tarih"])
              .groupby(["Olay_Ismi", "Hisse"], group_keys=False)[df.columns.tolist()]
              .apply(lambda g: g.fillna(method="ffill"))
        )
        df.dropna(subset=sayisal_sutunlar, inplace=True)
        return df

    @staticmethod
    def _son_hali_ver(df: pd.DataFrame) -> pd.DataFrame:
        sutun_siralamasi = [
            "Tarih", "Olay_Ismi", "Hisse", "Grup", "T0_Goreceli_Gun",
            "Duzeltilmis_Kapanis", "Log_Getiri", "Volatilite_10g", "Volatilite_30g"
        ]
        df = df[sutun_siralamasi].copy()
        df["Tarih"]               = pd.to_datetime(df["Tarih"]).dt.date
        df["Duzeltilmis_Kapanis"] = df["Duzeltilmis_Kapanis"].astype(float).round(6)
        df["Log_Getiri"]          = df["Log_Getiri"].astype(float).round(8)
        df["Volatilite_10g"]      = df["Volatilite_10g"].astype(float).round(8)
        df["Volatilite_30g"]      = df["Volatilite_30g"].astype(float).round(8)
        
        df.sort_values(["Olay_Ismi", "Hisse", "Tarih"], inplace=True)
        df.reset_index(drop=True, inplace=True)
        return df

print("Hücre 3 çalıştı: sınıf sisteme tanıtıldı.")

Hücre 3 çalıştı: sınıf sisteme tanıtıldı.


In [ ]:
# HÜCRE 4: İşlemi Başlatma ve Raporlama Fonksiyonu

def ozeti_yazdir(df: pd.DataFrame) -> None:
    ayirici = "─" * 70
    print(f"\n{ayirici}")
    print("  OLAY CALISMASI BORU HATTI — OZET RAPOR")
    print(ayirici)
    print(f"  Toplam satir   : {len(df):,}")
    print(f"  Toplam sutun   : {len(df.columns)}")
    print(f"\n  Olay basina satir:")
    for olay, sayi in df.groupby("Olay_Ismi").size().items():
        print(f"    {olay:<35} {sayi:>6,} satir")
    print(f"\n  Grup basina satir:")
    for grup, sayi in df.groupby("Grup").size().items():
        print(f"    {grup:<35} {sayi:>6,} satir")
    print(ayirici)

# calistir
ayarlar  = BoruHattiAyarlari()
muhendis = VeriMuhendisi(ayarlar)
sonuc_df = muhendis.calistir()

# Ozeti yazdir
ozeti_yazdir(sonuc_df)

In [5]:
#Görüntüleme

display(sonuc_df.head(10))

# display(sonuc_df[sonuc_df["Olay_Ismi"] == "ABD_Secimleri_2024"].head())

Price,Tarih,Olay_Ismi,Hisse,Grup,T0_Goreceli_Gun,Duzeltilmis_Kapanis,Log_Getiri,Volatilite_10g,Volatilite_30g
0,2015-11-16,ABD_Secimleri_2016,CL=F,Emtia_Makro,-358,41.740002,0.024249,0.024750,0.025493
1,2015-11-17,ABD_Secimleri_2016,CL=F,Emtia_Makro,-357,40.669998,-0.025969,0.018598,0.023880
2,2015-11-18,ABD_Secimleri_2016,CL=F,Emtia_Makro,-356,40.750000,0.001965,0.018342,0.023859
3,2015-11-19,ABD_Secimleri_2016,CL=F,Emtia_Makro,-355,40.540001,-0.005167,0.017990,0.022716
4,2015-11-20,ABD_Secimleri_2016,CL=F,Emtia_Makro,-354,40.389999,-0.003707,0.017786,0.022635
5,2015-11-23,ABD_Secimleri_2016,CL=F,Emtia_Makro,-351,41.750000,0.033117,0.022255,0.022087
6,2015-11-24,ABD_Secimleri_2016,CL=F,Emtia_Makro,-350,42.869999,0.026473,0.024151,0.022747
7,2015-11-25,ABD_Secimleri_2016,CL=F,Emtia_Makro,-349,43.040001,0.003958,0.022350,0.022777
8,2015-11-26,ABD_Secimleri_2016,CL=F,Emtia_Makro,-348,43.040001,0.000000,0.020075,0.022775
9,2015-11-27,ABD_Secimleri_2016,CL=F,Emtia_Makro,-347,41.709999,-0.031389,0.021213,0.023000
